In [1]:
import torch
import torch.nn.functional as F
import math
from typing import Optional

def multihead_attention(
    query: torch.Tensor,
    key: torch.Tensor,
    value: torch.Tensor,
    num_heads: int,
) -> torch.Tensor:
    batch_size, seq_len_q, d_model = query.shape
    seq_len_k = key.shape[1]
    
    assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
    
    d_head = d_model // num_heads
    q = query.view(batch_size, seq_len_q, num_heads, d_head).transpose(1, 2)
    k = key.view(batch_size, seq_len_k, num_heads, d_head).transpose(1, 2)
    v = value.view(batch_size, seq_len_k, num_heads, d_head).transpose(1, 2)
    
    scores = torch.matmul(q, k.transpose(-2, -1))
    scores = scores / math.sqrt(d_head)
    
    attention_weights = F.softmax(scores, dim=-1)
    output = torch.matmul(attention_weights, v)
    output = output.transpose(1, 2).contiguous().view(batch_size, seq_len_q, d_model)
    return output

In [2]:
from matdeeplearn.models.model_dev.sparse_attn.attention import SparseAttention

attn = SparseAttention(8, "strided", 32, 32)

In [3]:
%%timeit

query = torch.randn(20, 32, 512)
key = torch.randn(20, 32, 512)
value = torch.randn(20, 32, 512)
output, attn_matrix = attn(query, key, value)

AssertionError: 1, 32

In [ ]:
%%timeit

query = torch.randn(20, 32, 512)
key = torch.randn(20, 32, 512)
value = torch.randn(20, 32, 512)
output = multihead_attention(query, key, value, 8).shape

11.5 ms ± 890 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
